# Citrus: add a second 50,000 draft candidates
Resume the completed first 50,000 SQLite checkpoint. This notebook increases the cumulative target to 100,000 (30,000 knowledge / 70,000 practice), preserving existing candidates and exact duplicate checks. It generates only the remaining 50,000. It does not train a model.
Use a CUDA GPU and Internet. Multiple sessions may be required. All output is draft and needs factual, semantic and runtime review.


In [ ]:
%pip install -q "transformers>=4.51,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1" sentencepiece


In [ ]:
import pathlib, urllib.request
# Fixed repository revision makes the notebook reproducible.
REVISION = "3d5e8bbae98d7bef517d558611e79da2fec2414a"
BASE = f"https://raw.githubusercontent.com/gentle58715-ship-it/Citrus-Training-/{REVISION}"
ROOT = pathlib.Path("/kaggle/working/citrus-generator")
ROOT.mkdir(exist_ok=True)
for name in ["generate_citrus_kaggle.py", "audit_citrus_v3.py"]:
    urllib.request.urlretrieve(f"{BASE}/scripts/{name}", ROOT / name)


In [ ]:
import sqlite3, json
OUT = pathlib.Path("/kaggle/working/citrus-100k")
OUT.mkdir(exist_ok=True)
# REQUIRED on first run: attach completed first-batch output to Kaggle.
# For later sessions, attach the latest 100k-run checkpoint instead.
RESUME_DB = ""
if not (OUT / "progress.sqlite").exists():
    if not RESUME_DB:
        raise RuntimeError("Set RESUME_DB to the attached progress.sqlite; do not start batch 2 empty")
    src = sqlite3.connect(f"file:{RESUME_DB}?mode=ro", uri=True)
    counts = dict(src.execute("SELECT kind, COUNT(*) FROM records GROUP BY kind"))
    if counts.get("knowledge", 0) < 15000 or counts.get("practice", 0) < 35000:
        src.close()
        raise RuntimeError("First 50,000 is not complete. Resume the first notebook before starting batch 2")
    if counts.get("knowledge", 0) > 30000 or counts.get("practice", 0) > 70000:
        src.close()
        raise RuntimeError("Checkpoint already exceeds this notebook's 100k quotas")
    dst = sqlite3.connect(OUT / "progress.sqlite")
    src.backup(dst)
    dst.close(); src.close()


In [ ]:
import subprocess
MAX_MINUTES = 480
BASE_ARGS = ["python3", "-u", str(ROOT / "generate_citrus_kaggle.py"),
             "--out", str(OUT), "--knowledge", "30000", "--practice", "70000",
             "--extend-targets"]
try:
    subprocess.run(BASE_ARGS + ["--max-minutes", str(MAX_MINUTES)], check=True)
finally:
    subprocess.run(BASE_ARGS + ["--export-only"], check=True)


In [ ]:
import json, shutil
manifest = json.loads((OUT / "manifest.json").read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))
subprocess.run(["python3", str(ROOT / "audit_citrus_v3.py"), "--root", str(OUT)], check=True)
archive = shutil.make_archive("/kaggle/working/Citrus-100k-progress", "zip", OUT)
print("Saved:", archive)


## Output and remaining target
The final JSONL includes both batches: 100,000 candidates total. Do not concatenate it with the first 50k JSONL again. The SQLite checkpoint preserves normalized prompt/answer uniqueness across both batches. This is exact deduplication, not semantic deduplication.
All records are drafts, approved_total remains zero. Review before training and use family IDs for split assignment. The full project target remains 150,000; this notebook only extends to 100,000. No generation job has been launched by creating this notebook.
